In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, UnitNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img

In [ ]:
folder = Path("caltech101")
classes = sorted(d.name for d in folder.iterdir() if d.is_dir() and d.name not in ["BACKGROUND_Google", "train"])
rng = np.random.default_rng(42)
train_rows, test_rows = [], []
for name in classes:
    files = sorted((folder / name).glob("*.jpg"))
    rng.shuffle(files)
    train_rows.extend((str(f), name) for f in files[:30])
    test_rows.extend((str(f), name) for f in files[30:40])
train = pd.DataFrame(train_rows, columns=["filename", "class"])
test = pd.DataFrame(test_rows, columns=["filename", "class"])
train, val = train_test_split(train, test_size=0.2, stratify=train["class"], random_state=42)

In [ ]:
generator = ImageDataGenerator(preprocessing_function=preprocess_input)

In [ ]:
base = VGG16(weights="vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5",
             include_top=False, pooling="avg", input_shape=(224, 224, 3))
base.trainable = False
train_images = generator.flow_from_dataframe(train, x_col="filename", y_col="class", classes=classes,
    class_mode="sparse", target_size=(224, 224), batch_size=16, shuffle=False)
x_train = normalize(base.predict(train_images, verbose=0))
y_train = np.array(train_images.classes)

val_images = generator.flow_from_dataframe(val, x_col="filename", y_col="class", classes=classes,
    class_mode="sparse", target_size=(224, 224), batch_size=16, shuffle=False)
x_val = normalize(base.predict(val_images, verbose=0))
y_val = np.array(val_images.classes)

test_images = generator.flow_from_dataframe(test, x_col="filename", y_col="class", classes=classes,
    class_mode="sparse", target_size=(224, 224), batch_size=16, shuffle=False)

head = Sequential([Input((512,)), Dense(len(classes), activation="softmax")])
head.compile(optimizer=Adam(0.01), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
head.fit(x_train, y_train, epochs=40, batch_size=64, validation_data=(x_val, y_val))

In [ ]:
model = Sequential([base, UnitNormalization(), head])
base.trainable = True
for layer in base.layers[:-4]:
    layer.trainable = False
model.compile(optimizer=Adam(0.00001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_images, epochs=1, validation_data=val_images)

In [ ]:
loss, accuracy = model.evaluate(test_images)
print("Test loss:", loss)
print("Test accuracy (%):", round(100 * accuracy, 2))

In [ ]:
prediction = model.predict(test_images[0][0][:1], verbose=0).argmax()
plt.imshow(load_img(test["filename"].iloc[0]))
plt.title("Actual: " + test["class"].iloc[0] + " | Predicted: " + classes[prediction])
plt.axis("off")
plt.show()